In [2]:
# Necessary imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from datetime import datetime

from scipy import stats
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

In [3]:
columns = ['AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Required age', 'Price', 'Discount', 'DLC count', 'About the game', 'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux', 'Metacritic score', 'Metacritic url', 'User score', 'Positive', 'Negative', 'Score rank', 'Achievements', 'Recommendations', 'Notes', 'Average playtime forever', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks', 'Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 'Movies']

df = pd.read_csv("games.csv",names=columns, header=0)

In [ ]:
df.head()

In [ ]:
df.columns

### Data aggregation
For our analysis, we need a dataframe of developers, not of games. For this, let us group all games that share a developer, and look at the count of games released, overall engagement, and average price for each developer.

In [4]:
df['Release date'] = pd.to_datetime(df['Release date'])
df['Effort'] = (
    df['Screenshots'].fillna('').str.count(',') + 
    df['Screenshots'].notna().astype(int) +
    df['About the game'].fillna('').str.len() / 500
)

Making so games with DLC count >= 1 are counted as another game

In [5]:
df['game_weight'] = 1 # every entry defaults to 1

df.loc[df['DLC count'] >=1, 'game_weight'] = 2 # if a game has DLC count >= 1, increase that weight to 2

In [6]:
# Aggregate genres across all games by developer
def aggregate_genres(genres):
    unique_genres = set()
    for game_genres in genres.dropna():
        for genre in game_genres.split(','):
            unique_genres.add(genre.strip())
    return ','.join(sorted(unique_genres))
def aggregate_genres_count(genres):
    unique_genres = set()
    for game_genres in genres.dropna():
        for genre in game_genres.split(','):
            unique_genres.add(genre.strip())
    return len(unique_genres)

# Group by developer and aggregate features
developers_df = df.groupby('Developers').agg(
    game_count=('game_weight', 'sum'), # how many games released
    total_positive=('Positive', 'sum'), # total positive reviews
    total_negative=('Negative', 'sum'), # total negative reviews
    avg_price=('Price', 'mean'), # average game price
    max_success=('Positive', 'max'), # highest success in developers career
    genres=('Genres', aggregate_genres), # all the genres a developer has used
    genre_count=('Genres', aggregate_genres_count), # count of genres per developer
    start_year=('Release date', 'min'), # year of first game released
    end_year=('Release date', 'max'), # year of last game released
    avg_effort=('Effort', 'mean') # average effort put into steam pages for games
).reset_index()

In [ ]:
developers_df.head()


In [7]:
developers_df['engagement'] = developers_df['total_positive'] + developers_df['total_negative']
developers_df['avg_game_success'] =developers_df['engagement'] / developers_df['game_count']

developers_df.head()

,Developers,game_count,total_positive,total_negative,avg_price,max_success,genres,genre_count,start_year,end_year,avg_effort,engagement,avg_game_success
0,!CyberApex (SkagoGames),4,2,0,0.723333,1,"Action,Adventure,Casual,Indie,Simulation",5,2021-02-05,2025-07-01,8.656,2,0.5
1,!ReTigma Studio,1,2,0,15.000000,2,"Action,Adventure,Indie,RPG",4,2024-02-14,2024-02-14,5.764,2,2.0
2,#12,1,3,0,0.990000,3,"Action,Casual,Indie",3,2019-08-14,2019-08-14,6.796,3,3.0
3,#30A6D-S (Kuma),1,8,1,1.240000,8,"Adventure,Indie,RPG",3,2025-05-22,2025-05-22,17.824,9,9.0
4,#FF0000,1,42,2,1.990000,42,"Casual,Indie",2,2022-03-08,2022-03-08,14.088,44,44.0


In [8]:
def get_years_active(dataframe, end_year_column, start_year_column):
    years_active_list = []
    for row in dataframe.itertuples(index=True):
        index = row[0]
        start_value = start_year_column[index].year
        end_value = end_year_column[index].year
        difference = int(end_value) - int(start_value) + 1
        years_active_list.append(difference)
    dataframe['years_active'] = years_active_list

In [9]:
get_years_active(developers_df, developers_df['end_year'], developers_df['start_year'])
developers_df['frequency'] = developers_df['game_count'] / developers_df['years_active']

In [ ]:
developers_df.tail()

## Data Quality

In [ ]:
developers_df.isna().sum()

In [14]:
developers_df.duplicated().sum()

np.int64(0)

## Data Visualization

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    x='game_count',
    y='engagement',
    data=developers_df,
    s=25
)
plt.title('Game Count x Engagement')
plt.xlabel('Game Count')
plt.ylabel('Engagement')
plt.show()

## Jacob's graphs

In [ ]:
result = df.groupby("Developers").agg(
    positive=("Positive", "sum"),
    count=("Developers", "size")
)
maxX = round(result["count"].max())
result["count"].plot.hist(bins=range(1,maxX+1))
plt.yscale("log")
plt.xticks(range(1,maxX+1));

In [ ]:
freq = result['positive'].value_counts().sort_index()

plt.stem(freq.index, freq.values)
plt.yscale("log")
plt.xscale("log")

In [ ]:
dev_game_four = result[result["count"] == 10]
bins = [0] + [2**i for i in range(1, int(np.ceil(np.log2(dev_game_four["positive"].max() + 1))) + 1)]
counts, edges = np.histogram(dev_game_four["positive"], bins=bins)
plt.bar(
    [f"{int(a)}-{int(b)-1}" for a, b in zip(edges[:-1], edges[1:])],
    counts
)
plt.xlabel("Positive Interactions")
plt.ylabel("Number of developers")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [19]:
dev_game_four.shape

(101, 2)

## Outliers

In [ ]:
developers_df.describe()

In [ ]:
sns.boxplot(x=developers_df['game_count'])
plt.title("Box Plot for game_count")
plt.show()

In [ ]:
developers_df['game_count'].hist(bins=20, log=True)
plt.title("Histogram for game_count")
plt.show()

In [ ]:
developers_df.describe()

In [ ]:
z = np.abs(stats.zscore(developers_df['game_count']))
outliers_game_count = developers_df[z > 3]
outliers_game_count.shape

In [ ]:
outliers_game_count.head()

In [ ]:
outliers_game_count['game_count'].hist(bins=10)
plt.title("Histogram for outliers_game_count")
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    data=outliers_game_count,
    x='game_count',
    y='engagement',
    s=25
)
plt.title('outliers game_count vs engagement')
plt.xlabel('Game Count')
plt.ylabel('Engagement')
plt.show()

In [ ]:
z = np.abs(stats.zscore(developers_df['engagement']))
outliers_engagement = developers_df[z > 3]
outliers_engagement.shape


In [ ]:
outliers_engagement.head()

In [ ]:
Q1_game_count = developers_df['game_count'].quantile(0.25)
Q3_game_count = developers_df['game_count'].quantile(0.75)
IQR_game_count = Q3_game_count - Q1_game_count

lower_bound_game_count = Q1_game_count - 1.5 * IQR_game_count
upper_bound_game_count = Q3_game_count + 1.5 * IQR_game_count


outliers_game_count = developers_df[(developers_df['game_count'] < lower_bound_game_count) | (developers_df['game_count'] > upper_bound_game_count)]
outliers_game_count.shape

In [ ]:
Q1_engagement = developers_df['engagement'].quantile(0.25)
Q3_engagement = developers_df['engagement'].quantile(0.75)
IQR_engagement = Q3_engagement - Q1_engagement

lower_bound_engagement = Q1_engagement - 1.5 * IQR_engagement
upper_bound_engagement = Q3_engagement + 1.5 * IQR_engagement

outliers_engagement = developers_df[(developers_df['engagement'] < lower_bound_engagement) | (developers_df['engagement'] > upper_bound_engagement)]
outliers_engagement

In [ ]:
outliers_either = developers_df[(developers_df['engagement'] < lower_bound_engagement) | (developers_df['engagement'] > upper_bound_engagement) | (developers_df['game_count'] < lower_bound_game_count) | (developers_df['game_count'] > upper_bound_game_count)]

outliers_either.shape

In [ ]:
outliers_both = developers_df[((developers_df['engagement'] < lower_bound_engagement) | (developers_df['engagement'] > upper_bound_engagement)) & ((developers_df['game_count'] < lower_bound_game_count) | (developers_df['game_count'] > upper_bound_game_count))]

outliers_both.shape

## C. Preprocessing

In [ ]:
columns_to_drop = ['Estimated owners', 'Peak CCU', 'Required age', 'Discount', 'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 'Website', 'Support url', 'Support email','Windows', 'Mac', 'Linux', 'Metacritic score', 'Metacritic url', 'User score', 'Score rank', 'Achievements', 'Notes', 'Average playtime forever', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks', 'Publishers', 'Categories', 'Tags'  ]
essential_attributes_df = df.drop(columns=columns_to_drop)

essential_attributes_df.columns

### Processing missing entries

In [ ]:
essential_attributes_df.shape


In [36]:
essential_attributes_df = essential_attributes_df.dropna(subset=['Developers']) 

In [ ]:
essential_attributes_df['Developers'].isna().sum()

In [ ]:
essential_attributes_df['Developers'].isnull().sum()

In [ ]:
essential_attributes_df.shape


## D. Exploratory Data Analysis

## E. Pattern Discovery

In [ ]:
essential_attributes_df.shape

In [ ]:
essential_attributes_df.columns

In [ ]:
essential_attributes_df.columns

## K-Means on 2 features

To keep our data 2-dimensional, let us only first look at engagement (the sum of positive and negative reviews) and the game count

In [43]:
kmeans_df = developers_df[['game_count', 'engagement']].copy()
kmeans_df = kmeans_df.dropna() # dropping all missing entries for now, later we will revisit this

Scaling our features

In [44]:
scaler = StandardScaler()
kmeans_df_scaled = scaler.fit_transform(kmeans_df)

In [ ]:
inertia = []
K_range = range(2, 8)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    km.fit(kmeans_df_scaled)
    inertia.append(km.inertia_)

plt.plot(K_range, inertia, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.show()

For now, let us go with K = 4

In [ ]:
# initializing an instance of K-Means
kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init='auto'
)

# fitting the model and predicting cluster labels for each row
clusters = kmeans.fit_predict(kmeans_df_scaled)

# adding the cluster labels back to our developers dataframe
developers_df.loc[kmeans_df.index, 'Cluster'] = clusters
developers_df.head()

In [ ]:
developers_df.describe()

In [ ]:
cluster_summary = developers_df.groupby('Cluster')[['game_count','engagement']].mean()
print(cluster_summary)

In [ ]:
print(developers_df['Cluster'].value_counts())

In [ ]:
developers_df.head()

## Testing K-Means with all features

In [ ]:
developers_df.columns

In [52]:
scaler = StandardScaler()

kmeans_df = developers_df[['game_count', 'total_positive', 'total_negative', 'avg_price', 'max_success', 'genre_count', 'avg_game_success', 'years_active','frequency']].copy()

kmeans_df_scaled_all_features = scaler.fit_transform(kmeans_df)
# kmeans_df.head()

In [ ]:
inertia = []
K_range = range(2, 20)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    km.fit(kmeans_df_scaled_all_features)
    inertia.append(km.inertia_)

plt.plot(K_range, inertia, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.show()

In [ ]:
# initializing an instance of K-Means
kmeans = KMeans(
    n_clusters=6,
    random_state=42,
    n_init='auto'
)

# fitting the model and predicting cluster labels for each row
clusters = kmeans.fit_predict(kmeans_df_scaled_all_features)

# adding the cluster labels back to our developers dataframe
developers_df.loc[kmeans_df.index, 'Cluster'] = clusters
developers_df.head()


In [ ]:
cluster_summary = developers_df.groupby('Cluster')[['game_count', 'total_positive', 'total_negative', 'avg_price', 'max_success', 'genre_count', 'avg_game_success', 'years_active','frequency']].mean()
print(cluster_summary)

In [ ]:
print(developers_df['Cluster'].value_counts())

In [ ]:
whoareyou = developers_df[developers_df['Cluster'] == 1.0]
whoareyou.head()

In [ ]:
# reducing the scaled features to 2 dimensions
pca = PCA(n_components=2)
components = pca.fit_transform(kmeans_df_scaled_all_features)

# creating a temporary dataframe for plotting
plot_df = pd.DataFrame(components, columns=['PC1', 'PC2'])
plot_df['Cluster'] = developers_df.loc[kmeans_df.index, 'Cluster'].values

plt.figure(figsize=(10, 6))
sns.scatterplot(
    x='PC1', y='PC2', 
    hue='Cluster', 
    data=plot_df, 
    palette='viridis', 
    alpha=0.6,
    s=2
)
plt.title('K-Means Clusters Visualized')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Cluster')
plt.show()

In [ ]:
# sampling 10.000 developers
plot_sample = plot_df.sample(n=10000, random_state=52)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    x='PC1', y='PC2', 
    hue='Cluster', 
    data=plot_sample, 
    palette='viridis', 
    alpha=0.6,
    s=30
)
plt.title('Sample view of K-Means Clusters with PCA')
plt.show()

In [ ]:
regulars = developers_df[developers_df['Cluster'] != 2.0]
regulars = regulars[regulars['Cluster'] != 1.0]
print(regulars['Cluster'].value_counts())

## K-Means but no price

In [ ]:
scaler = StandardScaler()

regulars_kmeans = regulars[['game_count', 'total_positive', 'total_negative', 'max_success', 'avg_effort', 'avg_game_success', 'years_active','frequency']].copy()

kmeans_regulars_df_scaled_all_features = scaler.fit_transform(regulars_kmeans)

# initializing an instance of K-Means
kmeans = KMeans(
    n_clusters=6,
    random_state=42,
    n_init='auto'
)

# fitting the model and predicting cluster labels for each row
clusters = kmeans.fit_predict(kmeans_regulars_df_scaled_all_features)

# adding the cluster labels back to our developers dataframe
developers_df.loc[regulars_kmeans.index, 'Cluster'] = clusters
developers_df.head()


In [ ]:
cluster_summary = developers_df.groupby('Cluster')[['game_count', 'total_positive', 'total_negative', 'max_success', 'avg_effort', 'avg_game_success', 'years_active','frequency']].mean()
print(cluster_summary)

In [ ]:
interest = developers_df[developers_df['Cluster'] == 0.0]
interest['max_success'].max()

In [ ]:
interest.nlargest(5, 'total_positive')

In [ ]:
developers_df['Cluster'].value_counts()

### Adding a succesful tag to each developer 

In [10]:
developers_df['succesful'] = 0
developers_df.loc[developers_df['max_success'] >=1000, 'succesful'] = 1

In [ ]:
developers_df.head()

## Capping price and game count

In [11]:
developers_df['game_count'] = developers_df['game_count'].clip(upper=10)
developers_df['avg_price'] = developers_df['avg_price'].clip(upper=100)

In [ ]:
developers_df.head()

## Stuff done 11/11

In [70]:
new_df = developers_df[developers_df['Cluster'] != 1.0]
new_df = new_df[['Developers','game_count','engagement', 'Cluster']].copy()

In [ ]:
new_df

In [ ]:
# making a temporary df for the plotting
scaled_plot_df = pd.DataFrame(new_df, columns = ['scaled_game_count', 'scaled_engagement'])
scaled_plot_df['Cluster'] = new_df['Cluster'].values
scaled_plot_df.head()

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(
    x='game_count',
    y='engagement',
    hue='Cluster',
    data=new_df,
    s=25
)
plt.title('Developer Clusters')
plt.xlabel('Scaled Game Count')
plt.ylabel('Scaled Engagement')
plt.legend(title='Cluster')
plt.show()

In [ ]:
interest = new_df[new_df['Cluster'] == 0.0]
interest

In [ ]:
interest = pd.DataFrame(interest, columns = ['game_count','engagement'])
interest

In [ ]:
kmeans_10 = KMeans(
    n_clusters=10,
    random_state=21,
    n_init='auto'
)

interest_df_scaled = scaler.fit_transform(interest)
clusters = kmeans_10.fit_predict(interest_df_scaled)

# adding the cluster labels back to our developers dataframe
interest.loc[interest.index, 'Cluster'] = clusters
interest['log_engagement'] = np.log10(interest['engagement'])
interest.head()

In [ ]:
plt.figure(figsize=(10,6))

colors = ['red','green','blue','yellow','black','purple','brown','orange','cyan', 'grey']

sns.scatterplot(
    x='game_count',
    y='log_engagement',
    hue='Cluster',
    palette=colors,
    data=interest,
    s=25
)
plt.title('Developer Clusters')
plt.xlabel('Scaled Game Count')
plt.ylabel('Scaled Engagement')
plt.legend(title='Cluster')
plt.show()

In [ ]:
print(interest['Cluster'].value_counts())

Now let's take average price also into consideration

In [ ]:
# new df for k-means, but taking avg_price into consideration
kmeans_df_price = developers_df[['game_count', 'engagement', 'avg_price']].copy()

#applying scaling
kmeans_df_price_scaled = scaler.fit_transform(kmeans_df_price)

# using our k-means algorithm and passing our scaled np array
clusters_price = kmeans.fit_predict(kmeans_df_price_scaled)

# making a  copy of the developers df, but replacing the cluster labels
copy_df = developers_df.copy()
copy_df.loc[kmeans_df_price.index, 'Clusters'] = clusters_price

# turning the scaled np array into a proper dataframe
price_plot_df = pd.DataFrame(kmeans_df_price_scaled, columns = ['scaled_game_count', 'scaled_engagement','scaled_avg_price'])
price_plot_df['Cluster'] = copy_df['Cluster'].values
print(price_plot_df['Cluster'].value_counts())


To easily plot and visualize our clusters, we must apply PCA - Principal Component Analysis - to reduce our dimensionality

In [ ]:
# reducing the scaled features to 2 dimensions
pca = PCA(n_components=2)
components = pca.fit_transform(kmeans_df_price_scaled)

# creating a temporary dataframe for plotting
plot_df = pd.DataFrame(components, columns=['PC1', 'PC2'])
plot_df['Cluster'] = developers_df.loc[kmeans_df.index, 'Cluster'].values

plt.figure(figsize=(10, 6))
sns.scatterplot(
    x='PC1', y='PC2', 
    hue='Cluster', 
    data=plot_df, 
    palette='viridis', 
    alpha=0.6,
    s=2
)
plt.title('K-Means Clusters Visualized')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Cluster')
plt.show()

Let us try to zoom in a bit, by sampling a bit of this data

In [ ]:
# sampling 10.000 developers
plot_sample = plot_df.sample(n=10000, random_state=42)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    x='PC1', y='PC2', 
    hue='Cluster', 
    data=plot_sample, 
    palette='viridis', 
    alpha=0.6,
    s=30
)
plt.title('Sample view of K-Means Clusters with PCA')
plt.show()

It looks like we lost one of the clusters, as we saw previously that cluster 1.0 had only one developer

## F. Evaluation

## G. Decision Sensitivity Analysis

## H. Interpretation

# Begin Jacob Stuff (Apriori)

In [12]:
#-- Dependencies -----------------------------------
import pandas as pd
import matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

developers_df.head()

devTags = []
for row in developers_df.itertuples(index=False):
    tags = []
    if(row.succesful): tags.append('successful')
    for genre in row.genres.split(','): tags.append('Genre: '+genre)
    tags.append(str(row.start_year.year))
    if row.avg_effort > 15: tags.append("high effort")
    # if row.frequency > 1.5: tags.append('frequent')
    if row.game_count == 1: tags.append('Game Count: 1')
    elif row.game_count == 2: tags.append('Game Count: 2')
    elif row.game_count == 3: tags.append('Game Count: 3')
    elif row.game_count == 4: tags.append('Game Count: 4')
    elif row.game_count == 5: tags.append('Game Count: 5')
    elif row.game_count < 10: tags.append('Game Count: 6-9')
    else: tags.append('Game Count: 10+')
    devTags.append(tags)


In [13]:

te = TransactionEncoder()
te_array = te.fit(devTags).transform(devTags)
dev_encoded = pd.DataFrame(
    te_array,
    columns=te.columns_
)
dev_encoded.head()
frequent_itemsets = apriori(
    dev_encoded,
    min_support=0.01, # Only about 5.78% of developers are successful, we are selecting from within this subset
    use_colnames=True,
)

In [14]:

rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.0733 # tuned for 20 results
)
successful_rules = rules[
    rules['consequents'].apply(
        lambda x: 'successful' in x
    )
].sort_values(
    by='lift',
    ascending=False
)
len(successful_rules)
successful_rules.head(20)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
5594,"frozenset({Genre: Action, high effort})","frozenset({successful, Genre: Indie})",0.095568,0.041587,0.010157,0.106276,2.555500,1.0,0.006182,1.072381,0.673005,0.079974,0.067496,0.175250
2114,"frozenset({Genre: Action, high effort})",frozenset({successful}),0.095568,0.057769,0.013547,0.141749,2.453720,1.0,0.008026,1.097850,0.655058,0.096907,0.089129,0.188123
5591,"frozenset({Genre: Action, high effort, Genre: ...",frozenset({successful}),0.078549,0.057769,0.010157,0.129303,2.238269,1.0,0.005619,1.082157,0.600386,0.080505,0.075919,0.152558
2294,"frozenset({high effort, Genre: Adventure})",frozenset({successful}),0.088843,0.057769,0.011447,0.128843,2.230307,1.0,0.006314,1.081585,0.605418,0.084687,0.075431,0.163495
2117,frozenset({high effort}),"frozenset({Genre: Action, successful})",0.176629,0.034780,0.013547,0.076696,2.205205,1.0,0.007404,1.045398,0.663768,0.068466,0.043427,0.233099
380,frozenset({high effort}),frozenset({successful}),0.176629,0.057769,0.022276,0.126117,2.183122,1.0,0.012072,1.078212,0.658197,0.105015,0.072538,0.255860
2579,frozenset({high effort}),"frozenset({successful, Genre: Indie})",0.176629,0.041587,0.015962,0.090372,2.173079,1.0,0.008617,1.053632,0.655626,0.078922,0.050902,0.237100
2606,"frozenset({Genre: Simulation, Genre: Strategy})",frozenset({successful}),0.082927,0.057769,0.010335,0.124628,2.157340,1.0,0.005544,1.076377,0.584977,0.079280,0.070958,0.151765
2101,"frozenset({Genre: Action, Genre: Simulation})",frozenset({successful}),0.087841,0.057769,0.010870,0.123750,2.142148,1.0,0.005796,1.075299,0.584524,0.080676,0.070026,0.155959
5597,"frozenset({high effort, Genre: Indie})","frozenset({Genre: Action, successful})",0.137251,0.034780,0.010157,0.074000,2.127690,1.0,0.005383,1.042355,0.614323,0.062744,0.040634,0.183014


I removed the "frequent" tag because it dominated the results. In other words, being a frequent developer strongly suggested success. To understand why I thought this wasn't an interesting result, consider that (frequent+2 games)->(success). This suggested, to me, that a successful developer will quickly produce a second game.